Klasifikator cirilicnih slova

instalacija zavisnosti

In [2]:
!pip install numpy pillow
#TEMP_DOWNLOAD_DIR = 'temp_dataset_download'
#!pip install gdown


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


importi i konfiguracija

In [3]:
import numpy as np
import os
import pickle
from PIL import Image

KLASE = ['А', 'Б', 'В', 'Г', 'Д', 'Ђ', 'Е', 'Ж', 'З', 'И']

DATASET_DIR = 'dataset'   
IMG_SIZE    = 28          
TRAIN_SPLIT = 0.8         

LEARNING_RATE = 0.001
DECAY         = 1e-4
EPOCHS        = 1000
PRINT_EVERY   = 100


ucitavanje i preprocesiranje podataka

In [4]:
def ucitaj_dataset(dataset_dir=DATASET_DIR):
    X_lista, y_lista = [], []

    for i, slovo in enumerate(KLASE):
        putanja_do_foldera = os.path.join(dataset_dir, slovo)
        
        if not os.path.isdir(putanja_do_foldera):
            print(f'nema folder za {slovo} na {putanja_do_foldera}')
            continue

        sve_slike = []
        for f in os.listdir(putanja_do_foldera):
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                sve_slike.append(f)

        if len(sve_slike) == 0:
            print(f'folder {slovo} je prazan')
            continue

        for f in sve_slike:
            cela_putanja = os.path.join(putanja_do_foldera, f)
            try:
                slika = Image.open(cela_putanja).convert('L')
                slika = slika.resize((IMG_SIZE, IMG_SIZE))
                niz = np.array(slika, dtype=np.float32) / 255.0
                X_lista.append(niz.flatten())
                y_lista.append(i)
            except:
                print(f'ne moze da ucita {cela_putanja}')

    if len(X_lista) == 0:
        raise Exception('nijedna slika nije ucitana')

    return np.array(X_lista, dtype=np.float32), np.array(y_lista, dtype=np.int32)


def podeli_dataset(X, y, train_split=TRAIN_SPLIT, seed=42):
    np.random.seed(seed)
    indeksi = np.random.permutation(len(X))
    X = X[indeksi]
    y = y[indeksi]
    
    granica = int(len(X) * train_split)
    
    return X[:granica], y[:granica], X[granica:], y[granica:]


X, y = ucitaj_dataset()
X_train, y_train, X_test, y_test = podeli_dataset(X, y)

print(f'ukupno: {len(X)} slika')
print(f'trening: {len(X_train)}')
print(f'test: {len(X_test)}')
print()

for i in range(len(KLASE)):
    broj = sum(1 for v in y if v == i)
    print(f'{KLASE[i]}: {broj} primeraka')

ukupno: 2206 slika
trening: 1764
test: 442

А: 224 primeraka
Б: 220 primeraka
В: 220 primeraka
Г: 220 primeraka
Д: 220 primeraka
Ђ: 222 primeraka
Е: 220 primeraka
Ж: 220 primeraka
З: 220 primeraka
И: 220 primeraka


implementacija neuronske mreze

In [5]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases

    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)


class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0


class Layer_Dropout:
    def __init__(self, rate=0.2):
        self.rate = rate

    def forward(self, inputs):
        self.mask = np.random.binomial(1, 1 - self.rate, inputs.shape) / (1 - self.rate)
        self.output = inputs * self.mask

    def backward(self, dvalues):
        self.dinputs = dvalues * self.mask


class Activation_Softmax:
    def forward(self, inputs):
        najveci = np.max(inputs, axis=1, keepdims=True)
        exp_vrednosti = np.exp(inputs - najveci)
        self.output = exp_vrednosti / np.sum(exp_vrednosti, axis=1, keepdims=True)
        return self.output


class Loss_CategoricalCrossentropy:
    def forward(self, y_pred, y_true):
        broj_uzoraka = len(y_pred)
        klipuj = np.clip(y_pred, 1e-7, 1 - 1e-7)
        
        if y_true.ndim == 1:
            verovatnoce = klipuj[range(broj_uzoraka), y_true]
        else:
            verovatnoce = np.sum(klipuj * y_true, axis=1)
        
        gubitak = -np.log(verovatnoce)
        return np.mean(gubitak)
    
    def backward(self, y_pred, y_true):
        broj_uzoraka = len(y_pred)
        if y_true.ndim == 2:
            y_true = np.argmax(y_true, axis=1)
        
        self.dinputs = y_pred.copy()
        self.dinputs[range(broj_uzoraka), y_true] -= 1
        self.dinputs = self.dinputs / broj_uzoraka


class Optimizer_Adam:
    def __init__(self, learning_rate=0.001, decay=0.,
                 epsilon=1e-7, beta_1=0.9, beta_2=0.999):
        self.learning_rate = learning_rate
        self.current_learning_rate = learning_rate
        self.decay = decay
        self.iterations = 0
        self.epsilon = epsilon
        self.beta_1 = beta_1
        self.beta_2 = beta_2

    def pre_update_params(self):
        if self.decay != 0:
            self.current_learning_rate = self.learning_rate * (1.0 / (1.0 + self.decay * self.iterations))

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_momentums = np.zeros_like(layer.weights)
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_momentums = np.zeros_like(layer.biases)
            layer.bias_cache = np.zeros_like(layer.biases)

        layer.weight_momentums = self.beta_1 * layer.weight_momentums + (1 - self.beta_1) * layer.dweights
        layer.bias_momentums = self.beta_1 * layer.bias_momentums + (1 - self.beta_1) * layer.dbiases

        t = self.iterations + 1
        wm_korigovano = layer.weight_momentums / (1 - self.beta_1 ** t)
        bm_korigovano = layer.bias_momentums / (1 - self.beta_1 ** t)

        layer.weight_cache = self.beta_2 * layer.weight_cache + (1 - self.beta_2) * (layer.dweights ** 2)
        layer.bias_cache = self.beta_2 * layer.bias_cache + (1 - self.beta_2) * (layer.dbiases ** 2)

        wc_korigovano = layer.weight_cache / (1 - self.beta_2 ** t)
        bc_korigovano = layer.bias_cache / (1 - self.beta_2 ** t)

        layer.weights -= self.current_learning_rate * wm_korigovano / (np.sqrt(wc_korigovano) + self.epsilon)
        layer.biases -= self.current_learning_rate * bm_korigovano / (np.sqrt(bc_korigovano) + self.epsilon)

    def post_update_params(self):
        self.iterations += 1

inicijalizacija modela

In [6]:
np.random.seed(0)

dense1 = Layer_Dense(784, 128)
activation1 = Activation_ReLU()
dropout1 = Layer_Dropout(0.2)  

dense2 = Layer_Dense(128, 64)
activation2 = Activation_ReLU()
dropout2 = Layer_Dropout(0.2)  

dense3 = Layer_Dense(64, len(KLASE))
softmax_layer = Activation_Softmax()      
loss = Loss_CategoricalCrossentropy()
optimizer = Optimizer_Adam(learning_rate=0.001, decay=DECAY) 

print('model inicijalizovan')
print(f'ulaz:   784')
print(f'sloj 1: 128 neurona (ReLU, Dropout 0.2)')
print(f'sloj 2: 64 neurona (ReLU, Dropout 0.2)')
print(f'izlaz:   {len(KLASE)} neurona (Softmax)')
print(f'learning rate: 0.001')

model inicijalizovan
ulaz:   784
sloj 1: 128 neurona (ReLU, Dropout 0.2)
sloj 2: 64 neurona (ReLU, Dropout 0.2)
izlaz:   10 neurona (Softmax)
learning rate: 0.001


treniranje

In [7]:

for epoha in range(EPOCHS + 1):

    dense1.forward(X_train)
    activation1.forward(dense1.output)
    dropout1.forward(activation1.output)
    dense2.forward(dropout1.output)
    activation2.forward(dense2.output)
    dropout2.forward(activation2.output)
    dense3.forward(dropout2.output)
    softmax_layer.forward(dense3.output)                    
    gubitak = loss.forward(softmax_layer.output, y_train)  

    prognoze = np.argmax(softmax_layer.output, axis=1)     
    tacnost = np.mean(prognoze == y_train)

    loss.backward(softmax_layer.output, y_train)            
    softmax_layer.dinputs = loss.dinputs                    
    dense3.backward(softmax_layer.dinputs)                  
    dropout2.backward(dense3.dinputs)
    activation2.backward(dropout2.dinputs)
    dense2.backward(activation2.dinputs)
    dropout1.backward(dense2.dinputs)
    activation1.backward(dropout1.dinputs)
    dense1.backward(activation1.dinputs)

    optimizer.pre_update_params()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.update_params(dense3)
    optimizer.post_update_params()

    if epoha % PRINT_EVERY == 0:
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        softmax_layer.forward(dense3.output)
        test_prognoze = np.argmax(softmax_layer.output, axis=1)
        test_tacnost = np.mean(test_prognoze == y_test)
        
        print(f'epoha {epoha:5d} | gubitak: {gubitak:.4f} | '
              f'trening: {tacnost:.3f} | test: {test_tacnost:.3f} | lr: {optimizer.current_learning_rate:.6f}')

print(f'\nkonacna tacnost na treningu: {tacnost:.3f} ({tacnost*100:.1f}%)')
print(f'tacnost na test skupu: {test_tacnost:.3f} ({test_tacnost*100:.1f}%)')

epoha     0 | gubitak: 2.3026 | trening: 0.098 | test: 0.179 | lr: 0.001000
epoha   100 | gubitak: 0.6215 | trening: 0.807 | test: 0.738 | lr: 0.000990
epoha   200 | gubitak: 0.1536 | trening: 0.960 | test: 0.760 | lr: 0.000980
epoha   300 | gubitak: 0.0560 | trening: 0.989 | test: 0.778 | lr: 0.000971
epoha   400 | gubitak: 0.0286 | trening: 0.995 | test: 0.787 | lr: 0.000962
epoha   500 | gubitak: 0.0205 | trening: 0.998 | test: 0.790 | lr: 0.000952
epoha   600 | gubitak: 0.0108 | trening: 1.000 | test: 0.790 | lr: 0.000943
epoha   700 | gubitak: 0.0097 | trening: 0.998 | test: 0.785 | lr: 0.000935
epoha   800 | gubitak: 0.0079 | trening: 0.998 | test: 0.785 | lr: 0.000926
epoha   900 | gubitak: 0.0052 | trening: 0.999 | test: 0.783 | lr: 0.000917
epoha  1000 | gubitak: 0.0073 | trening: 0.999 | test: 0.783 | lr: 0.000909

konacna tacnost na treningu: 0.999 (99.9%)
tacnost na test skupu: 0.783 (78.3%)


evaluacija na test skupu

In [8]:

def softmax(x):
    exp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

def predvidi(X):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    activation2.forward(dense2.output)
    dense3.forward(activation2.output)
    probs = softmax(dense3.output)
    return np.argmax(probs, axis=1), probs


train_preds, _ = predvidi(X_train)
test_preds,  _ = predvidi(X_test)

train_acc = np.mean(train_preds == y_train)
test_acc  = np.mean(test_preds  == y_test)
print(f'trening tacnost: {train_acc:.3f}  ({train_acc*100:.1f}%)')
print(f'test tacnost:     {test_acc:.3f}  ({test_acc*100:.1f}%)')

print('\ntacnost po slovu (test skup):')
for i, s in enumerate(KLASE):
    mask = y_test == i
    if mask.sum() == 0:
        print(f'  {s}: nema primera u test skupu')
    else:
        acc_i = np.mean(test_preds[mask] == i)
        crta = '█' * int(acc_i * 20)
        print(f'  {s}: {acc_i*100:5.1f}%  {crta}')

trening tacnost: 1.000  (100.0%)
test tacnost:     0.783  (78.3%)

tacnost po slovu (test skup):
  А:  65.9%  █████████████
  Б:  68.6%  █████████████
  В:  77.1%  ███████████████
  Г:  86.7%  █████████████████
  Д:  76.2%  ███████████████
  Ђ:  70.2%  ██████████████
  Е:  86.4%  █████████████████
  Ж:  76.2%  ███████████████
  З:  86.4%  █████████████████
  И:  85.2%  █████████████████


cuvanje modela

In [9]:
model = {
    'w1': dense1.weights, 'b1': dense1.biases,
    'w2': dense2.weights, 'b2': dense2.biases,
    'w3': dense3.weights, 'b3': dense3.biases,
}
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('model sacuvan u model.pkl')

model sacuvan u model.pkl


testiranje nove slike

In [10]:
TEST_DIR = 'datasettest'

tacno = 0
pogresno = 0
rezultati = []

for tacna_klasa in sorted(os.listdir(TEST_DIR)):
    putanja_foldera = os.path.join(TEST_DIR, tacna_klasa)
    if not os.path.isdir(putanja_foldera):
        continue

    slike = [f for f in os.listdir(putanja_foldera)
             if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

    for ime in slike:
        puna_putanja = os.path.join(putanja_foldera, ime)
        slika = Image.open(puna_putanja).convert('L')
        slika = slika.resize((IMG_SIZE, IMG_SIZE))
        x = (np.array(slika, dtype=np.float32) / 255.0).flatten().reshape(1, -1)

        _, verovatnoce = predvidi(x)
        verovatnoce = verovatnoce[0]
        pred_indeks = np.argmax(verovatnoce)
        pred_slovo = KLASE[pred_indeks]
        procenat = verovatnoce[pred_indeks] * 100

        dobro = pred_slovo == tacna_klasa
        poruka = 'true' if dobro else f'false  (tacno: {tacna_klasa})'
        if dobro: 
            tacno += 1
        else:  
            pogresno += 1

        rezultati.append((tacna_klasa, ime, pred_slovo, procenat, dobro))
        print(f'  {tacna_klasa}/{ime:<20} → {pred_slovo}  {procenat:5.1f}%  {poruka}')

ukupno = tacno + pogresno
print(f'\n krajnji rezultat')
print(f'tacnih:    {tacno}/{ukupno}')
print(f'pogresnih: {pogresno}/{ukupno}')
print(f'procenat tacnosti: {tacno/ukupno*100:.1f}%')

print(f'\n po klasama')
for klasa in sorted(set(r[0] for r in rezultati)):
    redovi = [r for r in rezultati if r[0] == klasa]
    t = sum(1 for r in redovi if r[4])
    procenat_klase = t / len(redovi) * 100
    crtica = '█' * int(procenat_klase / 5)
    print(f'  {klasa}: {t:2d}/{len(redovi)}  {procenat_klase:5.1f}%  {crtica}')

  Ђ/Ђ_73_2022_0011.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0012.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0013.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0014.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0015.png   → Ђ   90.7%  true
  Ђ/Ђ_73_2022_0016.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0017.png   → Ђ   53.3%  true
  Ђ/Ђ_73_2022_0018.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0019.png   → Ђ   99.6%  true
  Ђ/Ђ_73_2022_0020.png   → Ђ   95.6%  true
  Ђ/Ђ_73_2022_0021.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0022.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0023.png   → Б  100.0%  false  (tacno: Ђ)
  Ђ/Ђ_73_2022_0024.png   → Ђ   39.4%  true
  Ђ/Ђ_73_2022_0025.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0026.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0027.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0028.png   → Ђ  100.0%  true
  Ђ/Ђ_73_2022_0029.png   → Ђ   98.5%  true
  Ђ/Ђ_73_2022_0030.png   → Ђ  100.0%  true
  А/А_52_2021_0011.png   → А  100.0%  true
  А/А_52_2021_0012.png   → А  100.0%  true
  А/А_52_2021_0013.png   → А  100.0%  tru